In [1]:
from pathlib import Path
from collections import defaultdict
import re   
import shutil
import time

import pandas as pd
import pyterrier as pt
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction


# The Excel file and this notebook must be in the same project folder.
PROJECT_FOLDER = Path.cwd()
EXCEL_FILE = PROJECT_FOLDER / "QQA23_TaskA_QPC_v1.1.xlsx"

# These folders are created by the notebook when they do not exist.
OUTPUT_FOLDER = PROJECT_FOLDER / "outputs"
TERRIER_FOLDER = PROJECT_FOLDER / "terrier_index"
CHROMA_FOLDER = PROJECT_FOLDER / "chroma_index"

# This folder is supplied by the user. The notebook only checks it.
LOCAL_MODELS_FOLDER = PROJECT_FOLDER / "my_local_models"
E5_MODEL_FOLDER = LOCAL_MODELS_FOLDER / "intfloatmultilingual-e5-large"
SILM_MODEL_FOLDER = LOCAL_MODELS_FOLDER / "multilingual-e5-small"

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
CHROMA_FOLDER.mkdir(parents=True, exist_ok=True)



if not pt.started():
    pt.init()


C:\Users\momoj\AppData\Local\Temp\ipykernel_9684\1136224169.py:32: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
c:\Users\momoj\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyterrier\java\mavenresolver.py:156: UserWarning: Attempted to get latest version of terrier-assemblies from maven, but was offline (https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/maven-metadata.xml [Errno 11001] getaddrinfo failed). Using latest cached version: 5.11
  warn(f'Attempted to get latest version of {packageName} from maven, but was offline ({url_str} {ue.reason}). Using latest cached version: {version}')
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
C:\Users\momoj\AppData\Local\Temp\ipykernel_9684\1136224169.py:33: Deprecat

In [ ]:
def repair_text(value):
    text = "" if pd.isna(value) else str(value)

    text = re.sub(
        r"_x([0-9A-Fa-f]{4})_",
        lambda match: chr(int(match.group(1), 16)),
        text,
    )

    try: 
        text = text.encode("latin1").decode("utf-8")                                # latin convert  to byte
    except UnicodeError:
        pass

    return text


arabic_marks = re.compile(                                                           # compile = select
    r"[ؐ-ًؚ-ٰٟۖ-ۭ]"
)


def normalize_arabic(text):
    text = str(text).replace("ـ", "")
    text = arabic_marks.sub("", text)
    text = text.translate(str.maketrans({
        "إ": "ا",
        "أ": "ا",
        "آ": "ا",
        "ٱ": "ا",
        "ى": "ي",
        "ة": "ه",
    }))
    text = re.sub(r"[^؀-ۿ\sA-Za-z0-9]", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()


raw_corpus = pd.read_excel(
    EXCEL_FILE,
    header=None,
    names=["docno", "text"],
)

raw_corpus["docno"] = raw_corpus["docno"].apply(repair_text)
raw_corpus["text"] = raw_corpus["text"].apply(repair_text)

corpus = raw_corpus.dropna(subset=["docno", "text"])
corpus = corpus[corpus["docno"].str.strip() != ""]
corpus = corpus[corpus["text"].str.strip() != ""]
corpus = corpus.drop_duplicates(subset=["docno"])
corpus = corpus.reset_index(drop=True)
corpus["indexed_text"] = corpus["text"].apply(normalize_arabic)

corpus.to_csv(
    OUTPUT_FOLDER / "quran_corpus_clean.csv",
    index=False,
    encoding="utf-8-sig",
)

In [3]:
if TERRIER_FOLDER.exists():
    shutil.rmtree(TERRIER_FOLDER)
TERRIER_FOLDER.mkdir(parents=True, exist_ok=True)

lexical_index_start = time.perf_counter()

terrier_index = pt.terrier.TerrierIndex(str(TERRIER_FOLDER))
terrier_indexer = terrier_index.indexer(
    tokeniser="whitespace",
    stemmer=None,
    stopwords=None,
)

terrier_documents = corpus[["docno", "indexed_text"]].rename(
    columns={"indexed_text": "text"}
)

index_reference = terrier_indexer.index(
    terrier_documents.to_dict(orient="records")
)

terrier = pt.IndexFactory.of(index_reference)
lexical_index_time = time.perf_counter() - lexical_index_start

In [4]:
bm25 = pt.terrier.Retriever(
    terrier,
    wmodel="BM25",
    num_results=5,
)

tfidf = pt.terrier.Retriever(
    terrier,
    wmodel="TF_IDF",
    num_results=5,
)

language_model = pt.terrier.Retriever(
    terrier,
    wmodel="Hiemstra_LM",
    num_results=5,
)

lexical_models = {
    "BM25": bm25,
    "TFIDF": tfidf,
    "LM": language_model,
}

In [ ]:
LOCAL_MODELS_FOLDER = PROJECT_FOLDER / r"C:\Users\momoj\Desktop\promaxir\my_local_models"
E5_MODEL_FOLDER = LOCAL_MODELS_FOLDER / r"C:\Users\momoj\Desktop\promaxir\my_local_models\intfloatmultilingual-e5-large"
SILM_MODEL_FOLDER = LOCAL_MODELS_FOLDER / r"C:\Users\momoj\Desktop\promaxir\my_local_models\multilingual-e5-small"


chroma_client = chromadb.PersistentClient(path=str(CHROMA_FOLDER))
vector_models = {}                      # for collectios 
vector_index_times = {}

local_model_paths = {
    "Vector_E5": E5_MODEL_FOLDER,
    "Vector_Silm": SILM_MODEL_FOLDER,
}

for model_name, model_folder in local_model_paths.items():
    try:
        chroma_client.delete_collection(model_name)
    except Exception:
        pass

    start = time.perf_counter()

    embedding_function = SentenceTransformerEmbeddingFunction(          # txt to vectors
        model_name=str(model_folder),
        device="cpu",
        normalize_embeddings=True,
    )

    collection = chroma_client.create_collection(
        name=model_name,
        embedding_function=embedding_function,
        metadata={"hnsw:space": "cosine"},
    )

    collection.add(
        ids=corpus["docno"].tolist(),
        documents=corpus["text"].tolist(),
    )

    vector_models[model_name] = collection
    vector_index_times[model_name] = time.perf_counter() - start

index_times = pd.DataFrame([
    {"model": "BM25", "indexing_time_seconds": lexical_index_time},
    {"model": "TFIDF", "indexing_time_seconds": lexical_index_time},
    {"model": "LM", "indexing_time_seconds": lexical_index_time},
    {"model": "Vector_E5", "indexing_time_seconds": vector_index_times["Vector_E5"]},
    {"model": "Vector_Silm", "indexing_time_seconds": vector_index_times["Vector_Silm"]},
])

index_times.to_csv(
    OUTPUT_FOLDER / "indexing_times.csv",
    index=False,
    encoding="utf-8-sig",
)


c:\Users\momoj\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2634.29it/s]


In [ ]:
MODEL_NAMES = [
    "BM25",
    "TFIDF",
    "LM",
    "Vector_E5",
    "Vector_Silm",
]


def search_with_pyterrier(model, query_text, top_k):
    query = pd.DataFrame({
        "qid": ["1"],
        "query": [query_text],
    })

    results = model.transform(query).head(top_k)

    return dict(
        zip(
            results["docno"].tolist(),
            (results["rank"] + 1).tolist(),
        )
    )


def search_with_chromadb(collection, query_text, top_k):                                #seach by vector
    results = collection.query(
        query_texts=[query_text],
        n_results=top_k,
        include=["documents", "distances"],
    )

    ids = results["ids"][0]

    return {
        document_id: rank + 1
        for rank, document_id in enumerate(ids)
    }


def compare_models_ranks(query_text, top_k=5):
    rank_results = defaultdict(dict)
    latency_rows = []

    for model_name, model in lexical_models.items():
        start = time.perf_counter()
        rank_results[model_name] = search_with_pyterrier(
            model,
            query_text,
            top_k,
        )
        latency_rows.append({
            "model": model_name,
            "query_latency_ms": (time.perf_counter() - start) * 1000,
        })

    for model_name, collection in vector_models.items():
        start = time.perf_counter()
        rank_results[model_name] = search_with_chromadb(
            collection,
            query_text,
            top_k,
        )
        latency_rows.append({
            "model": model_name,
            "query_latency_ms": (time.perf_counter() - start) * 1000,
        })

    all_docnos = []

    for model_name in MODEL_NAMES:
        for docno in rank_results[model_name]:
            if docno not in all_docnos:
                all_docnos.append(docno)

    rows = []

    for docno in all_docnos:
        document = corpus.loc[corpus["docno"] == docno].iloc[0]
        row = {
            "docno": docno,
            "text": document["text"],
        }

        for model_name in MODEL_NAMES:
            row[model_name] = rank_results[model_name].get(docno, "-")

        rows.append(row)

    return pd.DataFrame(rows), pd.DataFrame(latency_rows)

In [10]:
queries = {
    "Q1": "الجنة",
    "Q2": "ينطق",
    "Q3": "جسلي",
    "Q4": "الصيام عليكم كتب",
    "Q5": "للصلاة الوضوء",
    "Q6": "فضل الصبر وعاقبة الصابرين في البلاء",
    "Q7": "والزكاه الصاله",
    "Q8": "Patience and perseverance in times of hardship",
}

rank_tables = []
latency_tables = []

for query_type, query_text in queries.items():
    ranks, latencies = compare_models_ranks(query_text, top_k=5)

    ranks.insert(0, "query_type", query_type)
    ranks.insert(1, "query", query_text)

    latencies.insert(0, "query_type", query_type)
    latencies.insert(1, "query", query_text)

    rank_tables.append(ranks)
    latency_tables.append(latencies)

benchmark_ranks = pd.concat(rank_tables, ignore_index=True)
benchmark_latencies = pd.concat(latency_tables, ignore_index=True)

benchmark_ranks.to_csv(
    OUTPUT_FOLDER / "benchmark_ranks.csv",
    index=False,
    encoding="utf-8-sig",
)

benchmark_latencies.to_csv(
    OUTPUT_FOLDER / "benchmark_latencies.csv",
    index=False,
    encoding="utf-8-sig",
)

In [11]:
latency_summary = benchmark_latencies.groupby(
    "model",
    as_index=False,
)["query_latency_ms"].agg(
    mean_ms="mean",
    median_ms="median",
    min_ms="min",
    max_ms="max",
)

rank_summary_rows = []

for model_name in MODEL_NAMES:
    ranks = pd.to_numeric(
        benchmark_ranks[model_name],
        errors="coerce",
    )

    rank_summary_rows.append({
        "model": model_name,
        "retrieved_rows": int(ranks.notna().sum()),
        "mean_rank": ranks.mean(),
    })

rank_summary = pd.DataFrame(rank_summary_rows)

latency_summary.to_csv(
    OUTPUT_FOLDER / "latency_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

rank_summary.to_csv(
    OUTPUT_FOLDER / "rank_summary.csv",
    index=False,
    encoding="utf-8-sig",
)